# Heart Disease UCI - Exploratory Data Analysis

This notebook performs EDA on the Heart Disease UCI dataset (Cleveland subset).

**Dataset**: 303 patients, 13 features, binary target (disease present / absent).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

from configs.config import FEATURE_NAMES, TARGET_NAME, COLUMN_NAMES

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/raw/heart.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 2. Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print('Missing values per column:')
print(missing_df[missing_df['Count'] > 0])
print(f'\nTotal rows with any missing: {df.isnull().any(axis=1).sum()}')

## 3. Target Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

target_counts = df[TARGET_NAME].value_counts().sort_index()
labels = ['No Disease (0)', 'Disease (1)']

axes[0].bar(labels, target_counts.values, color=['#66c2a5', '#fc8d62'], edgecolor='black')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#66c2a5', '#fc8d62'], startangle=90,
            explode=(0.03, 0.03), textprops={'fontsize': 12})
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../screenshots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Distributions (Histograms)

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(COLUMN_NAMES):
    ax = axes[i]
    df[col].hist(bins=25, ax=ax, color='#8da0cb', edgecolor='black', alpha=0.8)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('')

for j in range(len(COLUMN_NAMES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../screenshots/feature_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
# Fill missing for correlation computation
df_clean = df.copy()
for col in df_clean.columns:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

plt.figure(figsize=(14, 10))
corr = df_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Box Plots by Target

In [ ]:
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(1, len(continuous_features), figsize=(20, 5))

for i, col in enumerate(continuous_features):
    sns.boxplot(data=df_clean, x=TARGET_NAME, y=col, ax=axes[i],
                palette=['#66c2a5', '#fc8d62'])
    axes[i].set_title(col, fontsize=13, fontweight='bold')
    axes[i].set_xticklabels(['No Disease', 'Disease'])
    axes[i].set_xlabel('')

plt.suptitle('Continuous Features by Target Class', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../screenshots/boxplots_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Categorical Features by Target

In [ ]:
cat_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(cat_features):
    ct = pd.crosstab(df_clean[col], df_clean[TARGET_NAME])
    ct.plot(kind='bar', ax=axes[i], color=['#66c2a5', '#fc8d62'], edgecolor='black')
    axes[i].set_title(col, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(['No Disease', 'Disease'], fontsize=9)
    axes[i].tick_params(axis='x', rotation=0)

axes[-1].set_visible(False)

plt.suptitle('Categorical Features by Target', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../screenshots/categorical_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Observations

- **Dataset size**: 303 samples, 13 features, 1 binary target.
- **Missing values**: `ca` and `thal` have a few missing entries (marked as `?`), handled via median imputation.
- **Class balance**: Roughly balanced (~54% no disease, ~46% disease).
- **Correlated features**: `cp`, `thalach`, `exang`, `oldpeak`, `slope`, `ca`, and `thal` show strong correlations with the target.
- **Age**: Disease patients tend to be slightly older.
- **Max heart rate (`thalach`)**: Healthy patients achieve higher max heart rates.
- **ST depression (`oldpeak`)**: Higher in disease patients.
- **Chest pain type (`cp`)**: Asymptomatic (type 4) is highly associated with disease.